# Centauro-Lite — varredura de experimentos

Treina **9 versoes** do modelo, uma por configuracao, e compara todas na mesma tabela.

A primeira rodada mostrou 0,9240 sem treino contra 0,6421 treinado — uma queda de 30,5%
com apenas 49 passos e 1 epoca. Isso e pouquissimo treino, e a pergunta obvia e o quanto
ainda sobra na mesa. Este notebook responde isso variando **uma coisa por vez**.

Uma variavel por linha nao e capricho: numa tabela onde duas coisas mudaram juntas,
nenhuma diferenca pode ser atribuida a nenhuma das duas.

**Antes de rodar, no painel da direita:**

1. **Accelerator** -> `GPU T4 x2`. Nao use a P100: o unsloth exige CUDA capability 7.0
   ou maior, a T4 e 7.5 e a P100 e 6.0.
2. **Internet** -> `On`.
3. **Persistence** -> `Files only`, para os modelos sobreviverem a queda da sessao.

**Quanto demora:** ~45 min por rodada na configuracao base, mais para as de 3 e 5
epocas. No total, algo entre 8 e 12 horas. A cota semanal do Kaggle e de 30 horas, entao
cabe — mas **provavelmente nao numa sessao so**, que expira em 12 horas.

Isso esta previsto: rodadas ja concluidas sao puladas. Se a sessao cair, abra de novo e
rode tudo outra vez; ele continua de onde parou em vez de repetir horas de trabalho
pronto.

Repositorio: https://github.com/Mathwesm/tcc_projeto.git

## 1. Ambiente

In [ ]:
# Uma T4 x2 expoe duas placas, e o unsloth nao lida bem com as duas ao mesmo tempo.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

In [ ]:
import os
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "tcc_projeto"

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/Mathwesm/tcc_projeto.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
os.environ["PYTHONUTF8"] = "1"
!pip install -q -e . --no-deps
!pip install -q pydantic pydantic-settings loguru typer pyyaml datasets pandas matplotlib
print("cwd:", Path.cwd())

## 2. O que vai ser testado

Cada arquivo em `configs/sweep/` e uma rodada. Rode esta celula para ver o que muda em
cada uma antes de gastar horas de GPU.

In [ ]:
from pathlib import Path

from centauro_lite.models.pipeline_config import PipelineConfig

default = PipelineConfig.from_yaml(Path("configs/default.yaml"))
print(f"{'rodada':<18} {'rank':>5} {'epocas':>7} {'lr':>9} {'modulos':>8}")
print("-" * 52)
print(
    f"{'(base)':<18} {default.model.lora_rank:>5} {default.training.num_epochs:>7.0f}"
    f" {default.training.learning_rate:>9.0e} {len(default.model.target_modules):>8}"
)
for path in sorted(Path("configs/sweep").glob("*.yaml")):
    config = PipelineConfig.from_yaml(path)
    print(
        f"{path.stem:<18} {config.model.lora_rank:>5} {config.training.num_epochs:>7.0f}"
        f" {config.training.learning_rate:>9.0e} {len(config.model.target_modules):>8}"
    )

# Todas as rodadas precisam usar os MESMOS dados, senao a tabela compara coisas
# diferentes. O fingerprint e o hash da configuracao de dados; se algum divergir, a
# suite de testes ja teria falhado, mas conferir aqui custa nada.
fingerprints = {
    PipelineConfig.from_yaml(p).data_fingerprint for p in Path("configs/sweep").glob("*.yaml")
}
print()
print("fingerprint dos dados:", fingerprints | {default.data_fingerprint})

## 3. Rodar a varredura

Cada rodada acontece num processo separado. Isso nao e cerimonia: carregar e descartar
modelos quantizados varias vezes no mesmo processo fragmenta a VRAM, e numa placa de
16 GB e a quarta rodada que morre. Processo novo por rodada tambem significa que um erro
custa uma linha da tabela, nao a varredura inteira.

Se a sessao cair no meio, **volte aqui e rode esta celula de novo**. As rodadas ja
medidas sao puladas.

In [ ]:
!python -m centauro_lite sweep --configs configs/sweep

## 4. A tabela

Aqui esta o resultado do TCC. A coluna `vs base` e a queda do NLL contra o modelo sem
treino, medida **dentro da mesma configuracao de dados** — comparar entre configuracoes
diferentes creditaria ao treino um ganho que veio de outra coisa.

In [ ]:
!python -m centauro_lite report

In [ ]:
from IPython.display import Image, display

display(Image("outputs/figures/ablation.png"))
display(Image("outputs/figures/per_experiment.png"))

## 5. O Minitaur, uma vez so

O Minitaur-8B nao entra na varredura: ele nao e treinado por nos e o numero dele nao muda
entre rodadas.

A celula abaixo roda o `prepare` de novo com `configs/minitaur.yaml`. Isso nao e
desperdicio: o dataset guarda ids de token, e um id pertence a um vocabulario so -- o id
2610 e "You" no Qwen3 e " askear" no Llama. Reaproveitar os dados do Qwen3 faria o
Minitaur ler ruido e devolver um numero plausivel sem levantar erro. O fingerprint do
split nao muda, entao sao exatamente os mesmos participantes; muda so a tokenizacao.

Ressalva para o texto do TCC: NLL por token nunca e perfeitamente comparavel entre
tokenizadores diferentes. Dar a cada modelo o seu vocabulario e o minimo, nao a solucao.

In [ ]:
import json
from pathlib import Path

results_path = Path("outputs/eval_results.json")
results = json.loads(results_path.read_text(encoding="utf-8")) if results_path.is_file() else {}

if "minitaur-8b" in results:
    print("Minitaur ja medido:", results["minitaur-8b"]["nll"])
else:
    # Duas etapas: o Minitaur precisa dos MESMOS participantes tokenizados com o
    # vocabulario dele. Sem isso ele leria ids do Qwen3 como se fossem seus e
    # devolveria um numero sobre ruido, sem erro nenhum.
    !python -m centauro_lite prepare --config configs/minitaur.yaml
    !python -m centauro_lite evaluate --config configs/minitaur.yaml --label "minitaur-8b"

## 6. Guardar

Use **Save Version** no canto superior direito. Sem isso os adapters e o
`eval_results.json` somem quando a sessao morrer.

Depois, me mande `outputs/report.txt` — e a tabela inteira em texto, mais facil de ler
que print de tela.

In [ ]:
!ls -lh outputs/
!cat outputs/report.txt